# Mean Squared Error (MSE)

**Mean Squared Error** measures the average squared difference between predicted values and actual (true) values. It is one of the most common loss/cost functions used in regression problems.

## Formula

$$ MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 $$

Where:
- $y_i$ = actual (true) value
- $\hat{y}_i$ = predicted value
- $n$ = number of samples

## Why square the errors?
- **No cancellation:** squaring makes every term positive, so positive and negative errors don't cancel each other out.
- **Penalizes large errors more:** a big miss is punished much more than a small one (error of 10 contributes 100, error of 1 contributes only 1).
- **Differentiable everywhere:** MSE is smooth, which makes it easy to optimize with calculus-based methods like gradient descent (unlike MAE, whose derivative is undefined at 0).

In [8]:
# NumPy is used for fast, vectorized array math
import numpy as np

## 1. Computing MSE manually, step by step

In [9]:
# Actual values and predicted values
y_true = np.array([3, 5, 2, 7])
y_pred = np.array([2.5, 5, 2, 8])

# Step 1: error (residual) for each point
errors = y_true - y_pred
print("Errors:", errors)

# Step 2: square each error
squared_errors = errors**2
print("Squared errors:", squared_errors)

# Step 3: take the mean
mse = np.mean(squared_errors)
print("MSE:", mse)

Errors: [ 0.5  0.   0.  -1. ]
Squared errors: [0.25 0.   0.   1.  ]
MSE: 0.3125


## 2. The one-line NumPy version

In practice you don't need the intermediate steps — NumPy lets you compute MSE in a single expression.

In [10]:
# Compute MSE directly: subtract, square, then average — no intermediate variables needed
mse = np.mean((y_true - y_pred) ** 2)
print("MSE:", mse)

MSE: 0.3125


## 3. Wrapping it in a reusable function

In [11]:
def mean_squared_error(y_true, y_pred):
    # Ensure inputs are NumPy arrays so element-wise math works even on plain lists
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    # Average of the squared differences between actual and predicted values
    return np.mean((y_true - y_pred) ** 2)


# Reuse the same y_true / y_pred from earlier to confirm the function matches the manual result
print(mean_squared_error(y_true, y_pred))

0.3125


## 4. Related metrics: RMSE and MAE

- **RMSE (Root Mean Squared Error):** the square root of MSE. It brings the error back into the same units as the original data, which makes it easier to interpret.

$$ RMSE = \sqrt{MSE} $$

- **MAE (Mean Absolute Error):** uses absolute differences instead of squared ones. It is less sensitive to outliers than MSE because large errors aren't amplified by squaring.

$$ MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i| $$

In [12]:
# RMSE: square root of MSE, brings error back to the original (unsquared) units
rmse = np.sqrt(mse)
# MAE: mean of absolute errors, an alternative that doesn't amplify large errors
mae = np.mean(np.abs(y_true - y_pred))

print("MSE :", mse)
print("RMSE:", rmse)
print("MAE :", mae)

MSE : 0.3125
RMSE: 0.5590169943749475
MAE : 0.375


## 5. Effect of outliers

Because MSE squares the errors, a single large mistake has an outsized effect on the score. Let's see this by adding one big outlier to the predictions.

In [13]:
# Same data as before, but with one prediction (last point) badly off — an outlier
y_true_2 = np.array([3, 5, 2, 7, 10])
y_pred_2 = np.array([2.5, 5, 2, 8, 25])  # last prediction is way off

# MSE squares the huge outlier error (15^2 = 225), so it dominates the average
print("MSE :", mean_squared_error(y_true_2, y_pred_2))
# MAE only takes the absolute value, so the same outlier has a much smaller relative impact
print("MAE :", np.mean(np.abs(y_true_2 - y_pred_2)))

MSE : 45.25
MAE : 3.3


Notice how MSE jumps much more dramatically than MAE once the outlier is introduced — this is the squaring effect in action. Choose MSE when large errors should be penalized heavily (e.g. safety-critical predictions), and MAE when you want a metric that's more robust to occasional outliers.

## 6. MSE with multi-dimensional arrays

NumPy's broadcasting makes it just as easy to compute MSE over 2D arrays, e.g. batches of predictions or multiple output features.

In [14]:
y_true_batch = np.array([[3, 5], [2, 7], [1, 9]])
y_pred_batch = np.array([[2.5, 5], [2, 8], [1.5, 8]])

# Overall MSE across every element
print("Overall MSE:", mean_squared_error(y_true_batch, y_pred_batch))

# Per-column (per-feature) MSE
per_feature_mse = np.mean((y_true_batch - y_pred_batch) ** 2, axis=0)
print("Per-feature MSE:", per_feature_mse)

# Per-row (per-sample) MSE
per_sample_mse = np.mean((y_true_batch - y_pred_batch) ** 2, axis=1)
print("Per-sample MSE:", per_sample_mse)

Overall MSE: 0.4166666666666667
Per-feature MSE: [0.16666667 0.66666667]
Per-sample MSE: [0.125 0.5   0.625]


## Summary

| Metric | Formula | Sensitive to outliers? | Units |
|---|---|---|---|
| MSE  | mean of squared errors | Very | squared units |
| RMSE | sqrt(MSE) | Very | original units |
| MAE  | mean of absolute errors | Less | original units |

MSE is widely used as a **loss function** during model training (it's smooth and differentiable, which suits gradient descent), while RMSE is often reported for **human-readable evaluation** since it's in the same units as the target variable.